# Eksperimen SML — Gregorius Willson

Dataset: **Telco Customer Churn**  
Tahapan: Data Loading → EDA → Preprocessing (mengikuti template eksperimen MSML).

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 2. Data Loading

In [ ]:
RAW_PATH = Path("../telco_raw/Telco-Customer-Churn.csv")
df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
df.describe(include="all")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicated rows:", df.duplicated().sum())
print("\nChurn distribution:\n", df["Churn"].value_counts(normalize=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["Churn"].value_counts().plot(kind="bar", ax=axes[0], color=["#4C78A8", "#F58518"])
axes[0].set_title("Churn Count")
axes[0].set_xlabel("Churn")

sns.histplot(df["tenure"], bins=30, ax=axes[1], kde=True)
axes[1].set_title("Tenure Distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[0])
axes[0].set_title("MonthlyCharges vs Churn")

contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index")
contract_churn.plot(kind="bar", stacked=True, ax=axes[1])
axes[1].set_title("Contract vs Churn (normalized)")
axes[1].legend(title="Churn")
plt.tight_layout()
plt.show()

In [ ]:
# TotalCharges sering bertipe object karena ada spasi kosong
blank_total = (df["TotalCharges"].astype(str).str.strip() == "").sum()
print("Blank TotalCharges:", blank_total)

## 4. Data Preprocessing

Langkah yang sama akan dikonversi ke `automate_Gregorius-Willson.py`.

In [ ]:
data = df.copy()

# Drop identifier
data = data.drop(columns=["customerID"])

# Bersihkan TotalCharges
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["TotalCharges"] = data["TotalCharges"].fillna(data["TotalCharges"].median())

# Target biner
data["Churn"] = data["Churn"].map({"Yes": 1, "No": 0})

# Encode kolom kategorikal
categorical_cols = data.select_dtypes(include=["object"]).columns.tolist()
for col in categorical_cols:
    encoder = LabelEncoder()
    data[col] = encoder.fit_transform(data[col].astype(str))

print("Processed shape:", data.shape)
data.head()

In [ ]:
print("Missing after preprocess:", data.isnull().sum().sum())
print("Feature dtypes:\n", data.dtypes.value_counts())
data.describe()

In [ ]:
output_dir = Path("telco_preprocessing")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "telco_preprocessing.csv"
data.to_csv(output_path, index=False)
print("Saved:", output_path.resolve())

## 5. Ringkasan

- Dataset Telco Customer Churn berhasil di-load.
- EDA menampilkan distribusi churn, tenure, charges, dan kontrak.
- Preprocessing: drop `customerID`, bersihkan `TotalCharges`, encode kategorikal, target biner.
- Output siap latih disimpan di `telco_preprocessing/telco_preprocessing.csv`.